# Baseline Model Evaluation

In this notebook, we'll establish baseline performance metrics for each model. These metrics will serve as a reference point to measure the impact of our optimization techniques.

## 1. Import Dependencies

In [ ]:
import os
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification
from transformers import AutoModelForQuestionAnswering, AutoModelForMaskedLM
import boto3

# Import utility functions
from utils import (
    measure_inference_time,
    get_model_size,
    measure_memory_usage,
    plot_comparison,
    estimate_monthly_cost,
    load_model_and_tokenizer,
    prepare_inputs as utils_prepare_inputs  # Rename to avoid conflict
)

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION

## 3. Load Model Information

In [ ]:
# Load model information from file
try:
    with open('model_info.json', 'r') as f:
        model_info = json.load(f)
    print(f"Loaded information for {len(model_info)} models")
except FileNotFoundError:
    print("model_info.json not found. Please run the first notebook to download models.")
    model_info = {}

## 4. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment_analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question_answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked_lm": "The [MASK] is a large language model trained by OpenAI."
}

## 5. Create Function to Prepare Inputs

In [ ]:
def prepare_inputs(task, tokenizer, sample_input):
    """Prepare inputs for different model tasks."""
    if task == "sequence-classification":
        inputs = tokenizer(sample_input, return_tensors="pt")
    elif task == "token-classification":
        inputs = tokenizer(sample_input, return_tensors="pt")
    elif task == "question-answering":
        inputs = tokenizer(
            sample_input["question"],
            sample_input["context"],
            return_tensors="pt"
        )
    elif task == "masked-lm":
        inputs = tokenizer(sample_input, return_tensors="pt")
        # Find the position of [MASK] token
        mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    return inputs

## 6. Create Function to Measure Baseline Metrics

In [ ]:
def measure_baseline_metrics(model_key, model_info, sample_inputs):
    """Measure baseline metrics for a model."""
    print(f"Measuring baseline metrics for {model_info['model_name']}...")
    
    # Load model and tokenizer from Hugging Face
    model_name = model_info["model_name"]
    task = model_info["task"]
    
    # Load model and tokenizer
    model, tokenizer = load_model_and_tokenizer(model_name, task)
    
    # Get sample input for this task
    sample_input = sample_inputs.get(model_key, "Sample input not available")
    
    # Prepare inputs
    inputs = prepare_inputs(task, tokenizer, sample_input)
    
    # Measure model size
    model_size = get_model_size(model)
    
    # Measure inference time
    inference_time = measure_inference_time(model, inputs)
    
    # Measure memory usage
    memory_usage = measure_memory_usage(model)
    
    # Get number of parameters
    num_parameters = sum(p.numel() for p in model.parameters())
    
    # Return metrics
    return {
        "model_key": model_key,
        "model_name": model_name,
        "task": task,
        "model_size": model_size,
        "inference_time": inference_time,
        "memory_usage": memory_usage,
        "num_parameters": num_parameters,
        "inputs": inputs,
        "outputs": None  # Will be filled in later
    }

## 7. Measure Baseline Metrics for All Models

In [ ]:
# Measure baseline metrics for all models
baseline_metrics = {}
for model_key, info in model_info.items():
    baseline_metrics[model_key] = measure_baseline_metrics(model_key, info, sample_inputs)
    print(f"Model: {info['model_name']}")
    print(f"Size: {baseline_metrics[model_key]['model_size']:.2f} MB")
    print(f"Inference Time: {baseline_metrics[model_key]['inference_time']:.2f} ms")
    print(f"Memory Usage: {baseline_metrics[model_key]['memory_usage']:.2f} MB")
    print(f"Number of Parameters: {baseline_metrics[model_key]['num_parameters']:,}")
    print("---")

In [ ]:
# Save baseline metrics (excluding outputs and inputs which aren't JSON serializable)
serializable_metrics = {}
for model_key, metrics in baseline_metrics.items():
    serializable_metrics[model_key] = {
        "model_key": metrics["model_key"],
        "model_name": metrics["model_name"],
        "task": metrics["task"],
        "model_size": metrics["model_size"],
        "inference_time": metrics["inference_time"],
        "memory_usage": metrics["memory_usage"],
        "num_parameters": metrics["num_parameters"]
    }

with open('baseline_metrics.json', 'w') as f:
    json.dump(serializable_metrics, f, indent=2)

print("Baseline metrics saved to baseline_metrics.json")

## 8. Display Baseline Metrics

In [ ]:
# Create a DataFrame with the metrics for better display
model_names = [metrics["model_name"] for metrics in serializable_metrics.values()]
model_sizes = [metrics["model_size"] for metrics in serializable_metrics.values()]
inference_times = [metrics["inference_time"] for metrics in serializable_metrics.values()]
memory_usages = [metrics["memory_usage"] for metrics in serializable_metrics.values()]
num_parameters = [metrics["num_parameters"] for metrics in serializable_metrics.values()]

# Create a DataFrame
metrics_df = pd.DataFrame({
    "Model": model_names,
    "Size (MB)": model_sizes,
    "Inference Time (ms)": inference_times,
    "Memory Usage (MB)": memory_usages,
    "Parameters": num_parameters
})

# Display the metrics table
metrics_df

## 9. Estimate Baseline Costs

In [ ]:
# Estimate costs for each model
cost_estimates = {}
for model_key, metrics in serializable_metrics.items():
    cost_estimates[model_key] = estimate_monthly_cost(metrics)

# Create a DataFrame for cost estimates
cost_data = []
for model_key, costs in cost_estimates.items():
    model_name = serializable_metrics[model_key]["model_name"]
    cost_data.append({
        "Model": model_name,
        "Compute Cost ($)": costs["compute_cost"],
        "Storage Cost ($)": costs["storage_cost"],
        "Total Cost ($)": costs["total_cost"]
    })

# Display the cost estimates table
cost_df = pd.DataFrame(cost_data)
cost_df

## 10. Next Steps

Now that we've established baseline metrics for our models, we're ready to apply optimization techniques. In the next notebook, we'll explore quantization to reduce model size and improve inference speed.